# Notebook 1: Data Cleaning
>


In [1]:
import pandas as pd
import numpy as np
import os

# If running from notebooks/ folder, adjust path to project root
os.chdir('..')  # go up one level so paths like data/raw/... work
print('Working directory:', os.getcwd())

Working directory: C:\Users\Ayush123\Desktop\data2\ride-demand-analysis


## Step 1 — Load Raw Data

In [2]:
rides   = pd.read_csv('data/raw/rides.csv',   parse_dates=['ride_date'])
drivers = pd.read_csv('data/raw/drivers.csv', parse_dates=['join_date'])
zones   = pd.read_csv('data/raw/zones.csv')

print('Rides shape  :', rides.shape)
print('Drivers shape:', drivers.shape)
print('Zones shape  :', zones.shape)
rides.head()

Rides shape  : (50000, 15)
Drivers shape: (5000, 5)
Zones shape  : (12, 4)


,ride_id,user_id,driver_id,pickup_zone,dropoff_zone,ride_date,ride_hour,day_of_week,is_weekend,distance_km,duration_minutes,fare_amount,payment_mode,ride_status,rating
0,1,16795,2513,Electronic City,BTM Layout,2024-11-23,16,Saturday,1,7.07,24,115.33,UPI,Completed,4.5
1,2,1860,186,MG Road,Electronic City,2024-02-27,18,Tuesday,0,1.40,4,45.59,Card,Completed,5.0
2,3,6390,4258,MG Road,BTM Layout,2024-01-13,8,Saturday,1,7.15,33,88.53,UPI,Completed,4.5
3,4,12964,3271,Hebbal,Hebbal,2024-05-20,17,Monday,0,7.30,24,78.11,UPI,Completed,4.5
4,5,12284,2449,Electronic City,BTM Layout,2024-05-05,10,Sunday,1,4.59,26,55.94,Card,Completed,4.5


## Step 2 — Data Quality Audit

In [3]:
print('=== Null Values ===')
print(rides.isnull().sum())

print('\n=== Data Types ===')
print(rides.dtypes)

print('\n=== Ride Status Breakdown ===')
print(rides['ride_status'].value_counts())

print('\n=== Fare Summary ===')
print(rides['fare_amount'].describe())

=== Null Values ===
ride_id             0
user_id             0
driver_id           0
pickup_zone         0
dropoff_zone        0
ride_date           0
ride_hour           0
day_of_week         0
is_weekend          0
distance_km         0
duration_minutes    0
fare_amount         0
payment_mode        0
ride_status         0
rating              0
dtype: int64

=== Data Types ===
ride_id                      int64
user_id                      int64
driver_id                    int64
pickup_zone                    str
dropoff_zone                   str
ride_date           datetime64[us]
ride_hour                    int64
day_of_week                    str
is_weekend                   int64
distance_km                float64
duration_minutes             int64
fare_amount                float64
payment_mode                   str
ride_status                    str
rating                     float64
dtype: object

=== Ride Status Breakdown ===
ride_status
Completed    39962
Cancelled     70

## Step 3 — Apply Cleaning Rules

In [4]:
before = len(rides)
rides.drop_duplicates(subset='ride_id', inplace=True)
print(f'Duplicates removed: {before - len(rides)}')

low = (rides['fare_amount'] < 20).sum()
rides.loc[rides['fare_amount'] < 20, 'fare_amount'] = 20.0
print(f'Low fares corrected: {low}')

assert rides['ride_hour'].between(0, 23).all(), 'Invalid hours found!'
print('Hour range: valid')

mask = rides['pickup_zone'] == rides['dropoff_zone']
print(f'Same-zone rides removed: {mask.sum()}')
rides = rides[~mask].copy()

Duplicates removed: 0
Low fares corrected: 0
Hour range: valid
Same-zone rides removed: 4159


## Step 4 — Feature Engineering

We create new columns that make analysis easier and more business-friendly.

In [5]:
def get_time_bucket(hour):
    if   0  <= hour < 6:  return 'Late Night'
    elif 6  <= hour < 10: return 'Morning Rush'
    elif 10 <= hour < 13: return 'Midday'
    elif 13 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening Rush'
    else:                 return 'Night'

rides['time_bucket'] = rides['ride_hour'].apply(get_time_bucket)
rides['month']       = rides['ride_date'].dt.month
rides['month_name']  = rides['ride_date'].dt.strftime('%b')
rides['week']        = rides['ride_date'].dt.isocalendar().week.astype(int)
rides['fare_per_km'] = (rides['fare_amount'] / rides['distance_km']).round(2)

print('New features added!')
print(rides[['time_bucket','month_name','fare_per_km']].head(5))

New features added!
    time_bucket month_name  fare_per_km
0     Afternoon        Nov        16.31
1  Evening Rush        Feb        32.56
2  Morning Rush        Jan        12.38
4        Midday        May        12.19
5        Midday        Apr        15.06


## Step 5 — Save Cleaned Data

In [6]:
os.makedirs('data/processed', exist_ok=True)
rides.to_csv('data/processed/rides_clean.csv', index=False)

print('Saved to data/processed/rides_clean.csv')
print(f'Final shape: {rides.shape}')
print(f'\nCompletion  : {(rides["ride_status"]=="Completed").mean():.1%}')
print(f'Cancellation: {(rides["ride_status"]=="Cancelled").mean():.1%}')
print('\nHead of cleaned data:')
rides.head()

Saved to data/processed/rides_clean.csv
Final shape: (45841, 20)

Completion  : 79.9%
Cancellation: 14.2%

Head of cleaned data:


,ride_id,user_id,driver_id,pickup_zone,dropoff_zone,ride_date,ride_hour,day_of_week,is_weekend,distance_km,duration_minutes,fare_amount,payment_mode,ride_status,rating,time_bucket,month,month_name,week,fare_per_km
0,1,16795,2513,Electronic City,BTM Layout,2024-11-23,16,Saturday,1,7.07,24,115.33,UPI,Completed,4.5,Afternoon,11,Nov,47,16.31
1,2,1860,186,MG Road,Electronic City,2024-02-27,18,Tuesday,0,1.40,4,45.59,Card,Completed,5.0,Evening Rush,2,Feb,9,32.56
2,3,6390,4258,MG Road,BTM Layout,2024-01-13,8,Saturday,1,7.15,33,88.53,UPI,Completed,4.5,Morning Rush,1,Jan,2,12.38
4,5,12284,2449,Electronic City,BTM Layout,2024-05-05,10,Sunday,1,4.59,26,55.94,Card,Completed,4.5,Midday,5,May,18,12.19
5,6,7265,3544,Jayanagar,Yeshwanthpur,2024-04-24,11,Wednesday,0,4.90,27,73.79,Cash,Completed,4.0,Midday,4,Apr,17,15.06
